# 🧠 Modelo CNN - Clasificación de Piezas Industriales
## Transfer Learning con VGG16

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**Objetivo:** Entrenar un modelo CNN usando Transfer Learning con VGG16 para clasificar 10 tipos de piezas industriales

---

## ⚙️ Configuración Inicial

In [ ]:
# Verificar entorno y GPU
import sys
import tensorflow as tf

IN_COLAB = 'google.colab' in sys.modules

print('✅ Ejecutando en Google Colab' if IN_COLAB else '⚠️ No estás en Colab')
print(f'🔥 TensorFlow versión: {tf.__version__}')
print(f'🚀 GPU disponible: {"SÍ ✅" if tf.config.list_physical_devices("GPU") else "NO ❌"}')

if tf.config.list_physical_devices('GPU'):
    print(f'💪 GPU: {tf.config.list_physical_devices("GPU")[0].name}')

## 📁 Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('\n✅ Google Drive montado correctamente')

## 📚 Importar Librerías

In [ ]:
# Librerías básicas
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime

# TensorFlow y Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
from tensorflow.keras.optimizers import Adam

# Métricas y visualización
from sklearn.metrics import classification_report, confusion_matrix
import itertools

# Configuración
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✅ Librerías importadas correctamente')

## 🔧 Configuración de Parámetros

In [ ]:
# Rutas
BASE_DIR = Path('/content/drive/MyDrive/datos')
DATA_DIR = BASE_DIR / 'DataSet'
TRAIN_DIR = DATA_DIR / 'Train_Dataset' / 'images'
VALID_DIR = DATA_DIR / 'Valid_Dataset' / 'images'
TEST_DIR = DATA_DIR / 'Test_Dataset' / 'images'

# Crear carpeta para modelos
MODEL_DIR = BASE_DIR / 'modelos'
MODEL_DIR.mkdir(exist_ok=True)

# Hiperparámetros
IMG_SIZE = 224  # VGG16 requiere 224x224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.0001
NUM_CLASSES = 10

# Verificar rutas
print(f'📁 Directorio de datos: {DATA_DIR}')
print(f'✅ TRAIN existe: {TRAIN_DIR.exists()}')
print(f'✅ VALID existe: {VALID_DIR.exists()}')
print(f'✅ TEST existe: {TEST_DIR.exists()}')
print(f'\n🎯 Configuración:')
print(f'   • Tamaño de imagen: {IMG_SIZE}x{IMG_SIZE}')
print(f'   • Batch size: {BATCH_SIZE}')
print(f'   • Épocas máximas: {EPOCHS}')
print(f'   • Learning rate: {LEARNING_RATE}')
print(f'   • Número de clases: {NUM_CLASSES}')

## 📊 Preparar Datos con Data Augmentation

Usaremos **ImageDataGenerator** para:
- Cargar imágenes eficientemente
- Aplicar Data Augmentation al conjunto de entrenamiento
- Normalizar valores de píxeles (0-1)

In [ ]:
# Data Augmentation para ENTRENAMIENTO
train_datagen = ImageDataGenerator(
    rescale=1./255,              # Normalizar a 0-1
    rotation_range=20,           # Rotación aleatoria ±20°
    width_shift_range=0.2,       # Desplazamiento horizontal
    height_shift_range=0.2,      # Desplazamiento vertical
    shear_range=0.2,             # Transformación de corte
    zoom_range=0.2,              # Zoom aleatorio
    horizontal_flip=True,        # Flip horizontal
    fill_mode='nearest'          # Rellenar píxeles vacíos
)

# SOLO normalización para VALIDACIÓN y TEST (sin augmentation)
valid_test_datagen = ImageDataGenerator(rescale=1./255)

print('✅ Data Augmentation configurado')
print('\n📊 Transformaciones aplicadas al entrenamiento:')
print('   • Normalización (0-1)')
print('   • Rotación ±20°')
print('   • Desplazamiento 20%')
print('   • Zoom ±20%')
print('   • Flip horizontal')
print('\n📊 Validación/Test:')
print('   • Solo normalización (0-1)')

## 🔄 Crear Generadores de Datos

**Importante:** Usamos `flow_from_directory()` que automáticamente:
- Lee las imágenes de subcarpetas
- Asigna etiquetas según el nombre de la carpeta
- Barajea los datos

In [ ]:
# Primero, necesitamos organizar las imágenes en carpetas por categoría
# Como están todas en una carpeta, usaremos el CSV para organizarlas

print('⏳ Preparando generadores de datos...')

# Leer CSV de labels
train_labels_df = pd.read_csv(DATA_DIR / 'Train_Dataset' / 'labels.csv')
valid_labels_df = pd.read_csv(DATA_DIR / 'Valid_Dataset' / 'labels.csv')
test_labels_df = pd.read_csv(DATA_DIR / 'Test_Dataset' / 'labels.csv')

# Preparar DataFrames con columnas correctas
# Columna 1 = filename, Columna 3 = category
train_df = pd.DataFrame({
    'filename': train_labels_df.iloc[:, 1],
    'category': train_labels_df.iloc[:, 3]
})

valid_df = pd.DataFrame({
    'filename': valid_labels_df.iloc[:, 1],
    'category': valid_labels_df.iloc[:, 3]
})

test_df = pd.DataFrame({
    'filename': test_labels_df.iloc[:, 1],
    'category': test_labels_df.iloc[:, 3]
})

print(f'✅ Train: {len(train_df)} imágenes')
print(f'✅ Valid: {len(valid_df)} imágenes')
print(f'✅ Test: {len(test_df)} imágenes')
print(f'\n🏭 Categorías: {train_df["category"].nunique()}')
print(train_df['category'].unique())

In [ ]:
# Crear generadores desde DataFrames
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=TRAIN_DIR,
    x_col='filename',
    y_col='category',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

validation_generator = valid_test_datagen.flow_from_dataframe(
    dataframe=valid_df,
    directory=VALID_DIR,
    x_col='filename',
    y_col='category',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = valid_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=TEST_DIR,
    x_col='filename',
    y_col='category',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('\n✅ Generadores creados exitosamente')
print(f'\n📊 Mapeo de clases:')
class_indices = train_generator.class_indices
for class_name, class_idx in class_indices.items():
    print(f'   {class_idx}: {class_name}')

## 🖼️ Visualizar Muestras con Augmentation

In [ ]:
# Visualizar el efecto del Data Augmentation
def plot_augmented_images(generator, n_images=8):
    images, labels = next(generator)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.ravel()
    
    class_names = list(generator.class_indices.keys())
    
    for i in range(n_images):
        axes[i].imshow(images[i])
        label_idx = np.argmax(labels[i])
        axes[i].set_title(f'{class_names[label_idx]}', fontsize=10)
        axes[i].axis('off')
    
    plt.suptitle('Muestras de Entrenamiento con Data Augmentation', 
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('🎨 Visualizando muestras con augmentation...')
plot_augmented_images(train_generator)

## 🧠 Construir Modelo con Transfer Learning (VGG16)

**Arquitectura:**
1. VGG16 preentrenado (frozen) - Extractor de características
2. GlobalAveragePooling2D - Reducir dimensionalidad
3. Dense(512) + ReLU + Dropout - Capa oculta
4. Dense(256) + ReLU + Dropout - Capa oculta
5. Dense(10) + Softmax - Capa de salida (10 clases)

In [ ]:
# Cargar VGG16 preentrenado (sin la capa top)
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Congelar las capas del modelo base
base_model.trainable = False

print('✅ VGG16 cargado (pesos de ImageNet)')
print(f'📊 Capas congeladas: {len(base_model.layers)}')

In [ ]:
# Construir el modelo completo
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

print('✅ Modelo construido')
print('\n📊 Arquitectura del modelo:')
model.summary()

## ⚙️ Compilar el Modelo

In [ ]:
# Compilar modelo
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('✅ Modelo compilado')
print(f'\n🎯 Configuración:')
print(f'   • Optimizer: Adam')
print(f'   • Learning rate: {LEARNING_RATE}')
print(f'   • Loss: categorical_crossentropy')
print(f'   • Metrics: accuracy')

## 📊 Configurar Callbacks

**Callbacks que usaremos:**
- **EarlyStopping:** Detiene el entrenamiento si no mejora
- **ModelCheckpoint:** Guarda el mejor modelo
- **ReduceLROnPlateau:** Reduce learning rate si se estanca

In [ ]:
# Crear nombre único para el modelo
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_name = f'vgg16_industrial_classifier_{timestamp}'
model_path = MODEL_DIR / f'{model_name}.keras'

# Callbacks
callbacks = [
    # Early Stopping: para si no mejora en 5 épocas
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Model Checkpoint: guarda el mejor modelo
    ModelCheckpoint(
        filepath=str(model_path),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    
    # Reduce LR: reduce learning rate si se estanca
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print('✅ Callbacks configurados')
print(f'\n💾 Modelo se guardará en:')
print(f'   {model_path}')

## 🚀 ENTRENAR EL MODELO

**⚠️ Este proceso tomará 15-25 minutos con GPU T4**

Podrás ver en tiempo real:
- Accuracy de entrenamiento y validación
- Loss de entrenamiento y validación
- Tiempo por época

In [ ]:
print('🚀 Iniciando entrenamiento...')
print(f'⏱️ Tiempo estimado: 15-25 minutos')
print('='*60)

# Entrenar
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

print('\n' + '='*60)
print('🎉 ¡Entrenamiento completado!')
print('='*60)

## 📈 Visualizar Resultados del Entrenamiento

In [ ]:
# Graficar accuracy y loss
def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    axes[0].set_title('Accuracy del Modelo', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Época')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Loss
    axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[1].set_title('Loss del Modelo', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Época')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Mostrar mejores resultados
    best_epoch = np.argmax(history.history['val_accuracy'])
    best_val_acc = history.history['val_accuracy'][best_epoch]
    best_train_acc = history.history['accuracy'][best_epoch]
    
    print('\n🏆 Mejores Resultados:')
    print(f'   • Época: {best_epoch + 1}')
    print(f'   • Train Accuracy: {best_train_acc:.4f} ({best_train_acc*100:.2f}%)')
    print(f'   • Validation Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')

plot_training_history(history)

## 🧪 Evaluar en Test Set

In [ ]:
# Evaluar en test set
print('🧪 Evaluando modelo en Test Set...')

test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)

print(f'\n📊 Resultados en Test Set:')
print(f'   • Test Loss: {test_loss:.4f}')
print(f'   • Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

## 🎯 Predicciones y Matriz de Confusión

In [ ]:
# Generar predicciones
print('🔮 Generando predicciones...')

test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

print('✅ Predicciones generadas')

In [ ]:
# Classification Report
class_names = list(test_generator.class_indices.keys())

print('📊 Classification Report:\n')
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Matriz de Confusión
def plot_confusion_matrix(y_true, y_pred, classes):
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(12, 10))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Matriz de Confusión', fontsize=16, fontweight='bold', pad=20)
    plt.colorbar()
    
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45, ha='right')
    plt.yticks(tick_marks, classes)
    
    # Añadir valores en las celdas
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
    
    plt.ylabel('Etiqueta Real', fontsize=12)
    plt.xlabel('Etiqueta Predicha', fontsize=12)
    plt.tight_layout()
    plt.show()

print('📊 Matriz de Confusión:')
plot_confusion_matrix(y_true, y_pred, class_names)

## 💾 Guardar Modelo y Metadatos

In [ ]:
# Guardar modelo en formato .h5 también (para compatibilidad)
model_h5_path = MODEL_DIR / f'{model_name}.h5'
model.save(str(model_h5_path))

print(f'✅ Modelo guardado en:')
print(f'   • {model_path} (Keras)')
print(f'   • {model_h5_path} (H5)')

In [ ]:
# Guardar metadatos del modelo
metadata = {
    'model_name': model_name,
    'timestamp': timestamp,
    'architecture': 'VGG16 Transfer Learning',
    'input_shape': [IMG_SIZE, IMG_SIZE, 3],
    'num_classes': NUM_CLASSES,
    'classes': class_names,
    'class_indices': class_indices,
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'epochs_trained': len(history.history['accuracy']),
        'learning_rate': LEARNING_RATE,
        'optimizer': 'Adam'
    },
    'performance': {
        'train_accuracy': float(history.history['accuracy'][-1]),
        'val_accuracy': float(history.history['val_accuracy'][-1]),
        'test_accuracy': float(test_accuracy),
        'test_loss': float(test_loss)
    },
    'data_augmentation': {
        'rotation_range': 20,
        'width_shift_range': 0.2,
        'height_shift_range': 0.2,
        'shear_range': 0.2,
        'zoom_range': 0.2,
        'horizontal_flip': True
    }
}

# Guardar JSON
metadata_path = MODEL_DIR / f'{model_name}_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f'\n✅ Metadatos guardados en:')
print(f'   • {metadata_path}')

## 📊 Resumen Final

In [ ]:
print('='*70)
print('🎉 ENTRENAMIENTO COMPLETADO - RESUMEN FINAL')
print('='*70)
print(f'\n🧠 Modelo: {model_name}')
print(f'\n📊 Performance:')
print(f'   • Train Accuracy: {history.history["accuracy"][-1]:.4f} ({history.history["accuracy"][-1]*100:.2f}%)')
print(f'   • Validation Accuracy: {history.history["val_accuracy"][-1]:.4f} ({history.history["val_accuracy"][-1]*100:.2f}%)')
print(f'   • Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')
print(f'\n📁 Archivos guardados:')
print(f'   • {model_name}.keras')
print(f'   • {model_name}.h5')
print(f'   • {model_name}_metadata.json')
print(f'\n🎯 Siguiente paso:')
print(f'   • Desplegar en AWS SageMaker')
print(f'   • Crear función Lambda para clasificación automática')
print(f'   • Configurar trigger S3')
print('='*70)

---
## 🎯 Próximos Pasos

1. ✅ **Modelo entrenado y guardado**
2. 📝 **Notebook 03:** Despliegue en AWS SageMaker
3. 📝 **Notebook 04:** Función Lambda + S3 Trigger
4. 📝 **Notebook 05:** Serie temporal - Predicción de acciones